In [ ]:
import torch
import torch.nn as nn
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Reynolds number
REYNOLDS_NUMBER = 100.0

# Points distribution
BOUNDARY_POINTS = 100
INTERIOR_POINTS = 98 * 98

# Boundary loss parameter
BOUNDARY_LOSS_PARAM = 1

In [ ]:
class LidCavityPINN(nn.Module):
    def __init__(self):
        super(LidCavityPINN, self).__init__()

        HIDDEN_LAYER_COUNT = 4
        NEURON_DENSITY = 64
        INPUT_DIM = 3
        OUTPUT_DIM = 3

        activation = torch.nn.Tanh()
        # activation = torch.nn.SiLU()
        # activation = SinActivation()

        self.hidden_layers = nn.Sequential(
            nn.Linear(INPUT_DIM, NEURON_DENSITY),
            activation,
        )
        for _ in range(HIDDEN_LAYER_COUNT):
            self.hidden_layers.append(nn.Linear(NEURON_DENSITY, NEURON_DENSITY))
            self.hidden_layers.append(activation)

        self.hidden_layers.append(nn.Linear(NEURON_DENSITY, OUTPUT_DIM))

        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_normal_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, x, y):
        out = self.hidden_layers(torch.cat([x, y], dim=1))
        return out[: 0:1], out[:, 1:2], out[:, 2:3]


model = LidCavityPINN().to(device)

In [ ]:
def generate_grid():
    grid = torch.rand(INTERIOR_POINTS, 2, device=device)
    return grid[:, 0:1], grid[:, 1:2]


def generate_boundary_points():
    sample = torch.linspace(0, 1, BOUNDARY_POINTS, device=device).unsqueeze(1)
    zeros = torch.zeros_like(sample)
    ones = torch.ones_like(sample)

    boundaries = [
        (sample, zeros, zeros, zeros),  # bottom
        (sample, ones, ones, zeros),  # top
        (zeros, sample, zeros, zeros),  # left
        (ones, sample, ones, zeros),  # right
    ]

    x, y, u, v = map(lambda t: torch.cat(t), boundaries)
    return x, y, u, v

In [ ]:
def boundary_loss_func(x, y, u_bc, v_bc):
    u, v, _ = model(x, y)
    return torch.mean((u - u_bc) ** 2) + torch.mean((v - v_bc) ** 2)


def pde_loss_func(x, y):
    x = x.requires_grad_(True)
    y = y.requires_grad_(True)
    u, v, p = model(x, y)

    def d_(f, v):
        return torch.autograd.grad(
            f, v, grad_outputs=torch.ones_like(f), create_graph=True, retain_graph=True
        )[0]

    u_x = d_(u, x)
    u_y = d_(u, y)

    v_x = d_(v, x)
    v_y = d_(v, y)

    p_x = d_(p, x)
    p_y = d_(p, y)

    u_xx = d_(u_x, x)
    u_yy = d_(u_y, y)

    v_xx = d_(v_x, x)
    v_yy = d_(v_y, y)

    continuity = u_x + v_y
    m_X = u * u_x + v * u_y + p_x - (1 / REYNOLDS_NUMBER) * (u_xx + u_yy)
    m_Y = u * v_x + v * v_y + p_y - (1 / REYNOLDS_NUMBER) * (v_xx + v_yy)

    return (
        torch.mean(continuity.pow(2)) + torch.mean(m_X.pow(2)) + torch.mean(m_Y.pow(2))
    )

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=5e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3000, gamma=0.5)

x_bc, y_bc, u_bc, v_bc = generate_boundary_points()

In [ ]:
import datetime

EPOCHS = 100

loss_history = {
    "pde": [],
    "boundary": [],
    "total": [],
}

start = datetime.datetime.now()
for i in range(EPOCHS):
    model.train()
    optimizer.zero_grad()

    x, y = generate_grid()
    pde_loss = pde_loss_func(x, y)
    bc_loss = boundary_loss_func(x_bc, y_bc, u_bc, v_bc)
    total_loss = pde_loss + BOUNDARY_LOSS_PARAM * bc_loss

    total_loss.backward()

    # remove exploding gradients
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()

    loss_history["pde"].append(pde_loss.item())
    loss_history["boundary"].append(bc_loss.item())
    loss_history["total"].append(total_loss.item())

    if i % 10 == 0:
        elapsed_time = datetime.datetime.now() - start
        start = datetime.datetime.now()
        lr_now = scheduler.get_last_lr()[0]
        print(
            f"Epoch {i}, Total Loss: {total_loss.item():.6f}, PDE Loss: {pde_loss.item():.6f}, BC Loss: {bc_loss.item():.6f}, LR: {lr_now:.6f}, Elapsed Time: {elapsed_time}"
        )

In [ ]:
@torch.no_grad()
def evaluate_on_grid(model, nx=150, ny=150):
    """Evaluate the trained model on a regular nx x ny grid."""
    xs = torch.linspace(0, 1, nx, device=device)
    ys = torch.linspace(0, 1, ny, device=device)
    X, Y = torch.meshgrid(xs, ys, indexing="ij")  # both (nx, ny)

    u, v, p = model(X.reshape(-1, 1), Y.reshape(-1, 1))

    U = u.reshape(nx, ny).cpu().numpy()
    V = v.reshape(nx, ny).cpu().numpy()
    P = p.reshape(nx, ny).cpu().numpy()
    return X.cpu().numpy(), Y.cpu().numpy(), U, V, P


def plot_results(X, Y, U, V, P, history):
    """
    Six-panel figure:
      Row 0: speed + streamlines | u-field | v-field
      Row 1: pressure            | vorticity | training loss
    """
    fig = plt.figure(figsize=(19, 11))
    fig.patch.set_facecolor("#10101e")
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.34)

    title_kw = dict(color="#dde4ff", fontsize=11, fontfamily="monospace")
    label_kw = dict(color="#8899bb", fontsize=9)

    speed = np.hypot(U, V)
    # Finite-difference vorticity: dv/dx - du/dy
    dx = X[1, 0] - X[0, 0]
    dy = Y[0, 1] - Y[0, 0]
    vorticity = np.gradient(V, axis=0) / dx - np.gradient(U, axis=1) / dy

    def styled_ax(ax, title):
        ax.set_facecolor("#1a1a2e")
        ax.set_title(title, **title_kw, pad=8)
        ax.tick_params(colors="#8899bb", labelsize=8)
        for spine in ax.spines.values():
            spine.set_edgecolor("#334466")
        ax.set_aspect("equal")
        ax.set_xlabel("x", **label_kw)
        ax.set_ylabel("y", **label_kw)

    # Panel 1: Speed + streamlines
    ax1 = fig.add_subplot(gs[0, 0])
    cf1 = ax1.contourf(X, Y, speed, levels=80, cmap="plasma")
    ax1.streamplot(
        X.T,
        Y.T,
        U.T,
        V.T,
        color="white",
        linewidth=0.55,
        density=1.5,
        arrowsize=0.7,
        arrowstyle="->",
    )
    cb1 = fig.colorbar(cf1, ax=ax1, fraction=0.046, pad=0.04)
    cb1.ax.tick_params(labelcolor="#9ab", labelsize=8)
    styled_ax(ax1, "Speed |u| + Streamlines")

    # Panel 2: u velocity
    ax2 = fig.add_subplot(gs[0, 1])
    cf2 = ax2.contourf(X, Y, U, levels=80, cmap="RdBu_r")
    cb2 = fig.colorbar(cf2, ax=ax2, fraction=0.046, pad=0.04)
    cb2.ax.tick_params(labelcolor="#9ab", labelsize=8)
    styled_ax(ax2, "Horizontal velocity  u")

    # Panel 3: v velocity
    ax3 = fig.add_subplot(gs[0, 2])
    cf3 = ax3.contourf(X, Y, V, levels=80, cmap="RdBu_r")
    cb3 = fig.colorbar(cf3, ax=ax3, fraction=0.046, pad=0.04)
    cb3.ax.tick_params(labelcolor="#9ab", labelsize=8)
    styled_ax(ax3, "Vertical velocity  v")

    # Panel 4: Pressure
    ax4 = fig.add_subplot(gs[1, 0])
    cf4 = ax4.contourf(X, Y, P, levels=80, cmap="coolwarm")
    ax4.contour(X, Y, P, levels=15, colors="white", linewidths=0.3, alpha=0.4)
    cb4 = fig.colorbar(cf4, ax=ax4, fraction=0.046, pad=0.04)
    cb4.ax.tick_params(labelcolor="#9ab", labelsize=8)
    styled_ax(ax4, "Pressure  p")

    # Panel 5: Vorticity
    ax5 = fig.add_subplot(gs[1, 1])
    vmax = np.percentile(np.abs(vorticity), 98)
    cf5 = ax5.contourf(
        X, Y, vorticity, levels=80, cmap="seismic", vmin=-vmax, vmax=vmax
    )
    cb5 = fig.colorbar(cf5, ax=ax5, fraction=0.046, pad=0.04)
    cb5.ax.tick_params(labelcolor="#9ab", labelsize=8)
    styled_ax(ax5, "Vorticity  w = dv/dx - du/dy")

    # Panel 6: Training loss
    ax6 = fig.add_subplot(gs[1, 2])
    ax6.set_facecolor("#1a1a2e")
    epochs = np.arange(1, len(history["total"]) + 1)
    ax6.semilogy(epochs, history["total"], color="#e06c75", lw=1.5, label="Total")
    ax6.semilogy(epochs, history["pde"], color="#61afef", lw=1.2, label="PDE")
    ax6.semilogy(epochs, history["bc"], color="#98c379", lw=1.2, label="BC")
    ax6.set_title("Training Loss", **title_kw, pad=8)  # type: ignore
    ax6.set_xlabel("Epoch", **label_kw)  # type: ignore
    ax6.set_ylabel("Loss (log scale)", **label_kw)  # type: ignore
    ax6.tick_params(colors="#8899bb", labelsize=8)
    ax6.legend(facecolor="#1a1a2e", edgecolor="#334", labelcolor="#dde4ff", fontsize=9)
    for spine in ax6.spines.values():
        spine.set_edgecolor("#334466")

    fig.suptitle(
        f"PINN -- 2D Lid-Driven Cavity  (Re = {REYNOLDS_NUMBER})",
        color="#e8f0ff",
        fontsize=15,
        fontfamily="monospace",
        y=0.98,
    )

    plt.savefig(
        "pinn_lid_cavity_results.png",
        dpi=150,
        bbox_inches="tight",
        facecolor=fig.get_facecolor(),
    )
    print("  Figure saved -> pinn_lid_cavity_results.png")
    plt.show()